# Regression Discontinuity — Simulation Companion
Julian Hsu

This notebook backs the *Regression Discontinuity Design* slides. It simulates a
retail RD design where an **eligibility score** `X` (0–100) crossing a **cutoff**
`c = 50` unlocks a **promotion** `W`, affecting **next-quarter purchases** `Y`.
The true effect is `tau = 5` (same convention as the Foundations deck).

We (1) hand-code the local-linear estimator to build intuition, (2) compare it to
`rdrobust`'s robust bias-corrected (RBC) estimator, (3) run the validation
battery (`rddensity` manipulation test, covariate continuity), and (4) run a
Monte-Carlo study that produces the bias and coverage tables shown on the slides.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdrobust import rdrobust, rdplot
from rddensity import rddensity

CUT, TAU = 50.0, 5.0

## The data-generating process
`ability` is an unobserved driver of the outcome. In the **manipulation**
scenario, high-`ability` customers just below the cutoff sort just above it —
this both creates a jump in the density of `X` and makes the just-above group
systematically different, which biases the RD estimate.

In [2]:
def baseline(x, shape="linear"):
    xc = (x - CUT) / 10.0
    if shape == "linear":
        return 10.0 + 0.8 * xc
    return 10.0 + 0.8 * xc + 2.6 * np.sin(0.9 * xc)   # high-curvature baseline


def gen(n=3000, shape="linear", manipulate=False, fuzzy=False, seed=None):
    rng = np.random.default_rng(seed)
    x = np.clip(rng.normal(CUT, 18, n), 0, 100)
    ability = rng.normal(0, 1, n)
    if manipulate:
        below = (x > CUT - 6) & (x < CUT)
        move = below & (ability > 0.2) & (rng.random(n) < 0.75)
        x[move] = CUT + rng.uniform(0, 4, move.sum())
    if fuzzy:
        w = (rng.random(n) < np.where(x >= CUT, 0.75, 0.20)).astype(float)
    else:
        w = (x >= CUT).astype(float)
    y = baseline(x, shape) + TAU * w + 2.0 * ability + rng.normal(0, 1.5, n)
    z = 0.5 + 0.03 * (x - CUT) + 0.8 * ability + rng.normal(0, 0.6, n)  # predetermined covariate
    return x, w, y, z

## Estimators
`naive` ignores the design; `global_poly` fits a global polynomial with
side-specific slopes; `local_linear` is a triangular-kernel local-linear fit
whose treatment effect is the difference of the two fitted intercepts at `c`.

In [3]:
def naive(x, w, y):
    return y[w == 1].mean() - y[w == 0].mean()


def global_poly(x, w, y, order=4):
    xc = x - CUT
    feats = [np.ones_like(xc), w]
    for k in range(1, order + 1):
        feats += [xc ** k, w * xc ** k]
    beta, *_ = np.linalg.lstsq(np.column_stack(feats), y, rcond=None)
    return beta[1]


def local_linear(x, w, y, h=8.0, return_se=False):
    xc = x - CUT
    ints, ses = {}, {}
    for side, mask in (("r", w == 1), ("l", w == 0)):
        xs, ys = xc[mask], y[mask]
        sel = np.abs(xs) <= h
        xs, ys = xs[sel], ys[sel]
        wgt = np.maximum(1 - np.abs(xs / h), 0)            # triangular kernel
        X = np.column_stack([np.ones_like(xs), xs]); W = np.diag(wgt)
        XtW = X.T @ W
        beta = np.linalg.solve(XtW @ X, XtW @ ys)
        ints[side] = beta[0]
        resid = ys - X @ beta
        s2 = (wgt * resid ** 2).sum() / max(wgt.sum() - 2, 1)
        cov = np.linalg.inv(XtW @ X) @ (XtW @ W @ X) @ np.linalg.inv(XtW @ X) * s2
        ses[side] = np.sqrt(max(cov[0, 0], 0))
    tau = ints["r"] - ints["l"]
    return (tau, np.sqrt(ses["r"] ** 2 + ses["l"] ** 2)) if return_se else tau

## One clean dataset: the estimators side by side
The naive difference is biased upward (it compares high-score to low-score
customers); the local methods recover `tau = 5`. `rdrobust` reports Conventional,
Bias-Corrected, and **Robust** rows — report the Robust interval.

In [4]:
x, w, y, z = gen(4000, "linear", seed=7)
print(f"naive mean difference : {naive(x, w, y):+.3f}")
print(f"global polynomial (4) : {global_poly(x, w, y):+.3f}")
t, se = local_linear(x, w, y, 8.0, return_se=True)
print(f"local linear (h=8)    : {t:+.3f}   95% CI [{t-1.96*se:+.3f}, {t+1.96*se:+.3f}]")

out = rdrobust(y=y, x=x, c=CUT)
print(f"\nrdrobust MSE-optimal bandwidth h = {out.bws.loc['h','left']:.2f}")
print(out.coef.round(3).join(out.ci.round(3)))

naive mean difference : +7.337
global polynomial (4) : +4.952
local linear (h=8)    : +5.010   95% CI [+4.448, +5.573]

rdrobust MSE-optimal bandwidth h = 10.73
                Coeff  CI Lower  CI Upper
Conventional    4.951     4.438     5.464
Bias-Corrected  4.937     4.425     5.450
Robust          4.937     4.324     5.551


## The RD plot
`rdplot` bins the running variable and overlays polynomial fits on each side —
the first thing to look at in any RD analysis.

In [5]:
_ = rdplot(y=y, x=x, c=CUT, title="RD plot: next-quarter purchases vs. eligibility score",
           x_label="Eligibility score X", y_label="Next-quarter purchases Y")

## Validation 1 — manipulation (density) test
`rddensity` implements the local-polynomial density test (Cattaneo, Jansson, Ma
2020), a modern version of McCrary (2008). The null is a **continuous** density
at the cutoff. Under manipulation the p-value drops and we reject continuity.

In [6]:
xc_clean, *_ = gen(6000, "linear", manipulate=False, seed=3)
xc_manip, *_ = gen(6000, "linear", manipulate=True, seed=3)
for label, xx in [("clean", xc_clean), ("manipulated", xc_manip)]:
    p = float(rddensity(X=xx, c=CUT).test["p_jk"])
    print(f"{label:12s}: density-test p-value = {p:.4f}  ->  "
          f"{'reject continuity (manipulation!)' if p < 0.05 else 'no evidence of manipulation'}")

clean       : density-test p-value = 0.1750  ->  no evidence of manipulation
manipulated : density-test p-value = 0.0043  ->  reject continuity (manipulation!)


## Validation 2 — covariate continuity (placebo outcome)
A predetermined covariate `Z` should **not** jump at the cutoff. We simply run
the RD with `Z` as the outcome; the estimate should be indistinguishable from 0
in a clean design, but jumps once high-ability units sort across the cutoff.

In [7]:
for label, man in [("clean", False), ("manipulated", True)]:
    x, w, y, z = gen(6000, "linear", manipulate=man, seed=4)
    o = rdrobust(y=z, x=x, c=CUT)
    coef = float(o.coef.loc["Robust", "Coeff"])
    lo, hi = float(o.ci.loc["Robust", "CI Lower"]), float(o.ci.loc["Robust", "CI Upper"])
    print(f"{label:12s}: covariate jump = {coef:+.3f}  95% RBC CI [{lo:+.3f}, {hi:+.3f}]  "
          f"{'<- jumps! design suspect' if lo > 0 or hi < 0 else '(continuous, OK)'}")

clean       : covariate jump = -0.121  95% RBC CI [-0.292, +0.050]  (continuous, OK)


manipulated : covariate jump = +0.739  95% RBC CI [+0.542, +0.936]  <- jumps! design suspect


## Monte-Carlo study: bias and coverage
We repeat the experiment over many datasets and three scenarios (**clean**,
**nonlinear**, **manipulation**), recording the mean bias and the 95%-CI coverage
for each estimator. These are the numbers reported on the slides.

In [8]:
def mc(shape, manipulate, R=500, n=3000):
    rows = {k: [] for k in ["naive", "gpoly", "local", "rdr"]}
    cov = {"local": [], "rdr": []}
    for r in range(R):
        x, w, y, z = gen(n, shape, manipulate=manipulate, seed=1000 + r)
        rows["naive"].append(naive(x, w, y))
        rows["gpoly"].append(global_poly(x, w, y))
        t, se = local_linear(x, w, y, 8.0, return_se=True)
        rows["local"].append(t); cov["local"].append(abs(t - TAU) <= 1.96 * se)
        try:
            o = rdrobust(y=y, x=x, c=CUT)
            rc = float(o.coef.loc["Robust", "Coeff"])
            lo = float(o.ci.loc["Robust", "CI Lower"]); hi = float(o.ci.loc["Robust", "CI Upper"])
            rows["rdr"].append(rc); cov["rdr"].append(lo <= TAU <= hi)
        except Exception:
            pass
    bias = {k: np.mean(v) - TAU for k, v in rows.items()}
    coverage = {k: np.mean(v) for k, v in cov.items()}
    return bias, coverage


scenarios = [("Clean", "linear", False), ("Nonlinear", "nonlinear", False),
             ("Manipulation", "linear", True)]
B, C = {}, {}
for name, shape, man in scenarios:
    B[name], C[name] = mc(shape, man)

labels = {"naive": "Naive mean difference", "gpoly": "Global polynomial (order 4)",
          "local": "Local linear (hand-coded)", "rdr": "rdrobust (RBC)"}
bias_tbl = pd.DataFrame({sc: {labels[k]: B[sc][k] for k in labels} for sc, *_ in scenarios})
cov_tbl = pd.DataFrame({sc: {"Local linear (naive SE)": C[sc]["local"],
                             "rdrobust (RBC)": C[sc]["rdr"]} for sc, *_ in scenarios})
print("Mean bias (tau_hat - tau),  tau = 5:")
print(bias_tbl.round(2))
print("\n95% CI coverage:")
print(cov_tbl.round(2))

Mean bias (tau_hat - tau),  tau = 5:
                             Clean  Nonlinear  Manipulation
Naive mean difference         2.28       5.33          2.60
Global polynomial (order 4)   0.01      -0.04          1.99
Local linear (hand-coded)     0.02       0.05          1.73
rdrobust (RBC)                0.00      -0.06          1.89

95% CI coverage:
                         Clean  Nonlinear  Manipulation
Local linear (naive SE)   0.95       0.94           0.0
rdrobust (RBC)            0.94       0.95           0.0


## Takeaways
* Local methods (local linear, `rdrobust`) recover `tau` even when the baseline
  is curved; the naive difference does not.
* **Robust bias-corrected** intervals from `rdrobust` hold their nominal 95%
  coverage in the clean and nonlinear scenarios.
* Under **manipulation**, *every* estimator is biased and coverage collapses —
  which is exactly why the density and covariate tests come first. A broken
  design cannot be fixed by a better estimator.